In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
from sklearn.metrics import r2_score
from srkan import regressor, SympyEvaluator
jax.config.update("jax_enable_x64", True)


In [2]:
key = jr.key(42)
x_np = np.random.uniform(0.1, 3.0, (1000, 2))
y_np = (x_np[:, 0] / (1 + x_np[:, 1]**2)) #+0.01*np.random.normal(size=(1000,)) # Add noise

x = jnp.array(x_np)
y = jnp.array(y_np).reshape(-1, 1)


In [3]:
model = regressor(
    key=jr.key(0),
    result_threshold=1e-3,        # Stop if MSE < 1e-3 (change in the case of added noise)
    functions=["all"],            # Use all available library functions
    do_rounding=True,             # Round constants (e.g., 0.999 -> 1.0)
    backward_elim=True,           # Prune unnecessary terms
    manipulate_output=["inv"],    # Try fitting 1/y to handle rational functions
    combination_kan_types=[["mult", "mult"]], # Search strategy
    verbosity=1
)

print("Starting Symbolic Regression...")
expression = model.fit(x, y)

# 4. Evaluate and Print Results
print(f"\n recovered Expression: {expression}")

# Evaluate the expression numerically on the data
evaluator = SympyEvaluator(expression)
y_pred = evaluator(x, None, mse=False)

r2 = r2_score(y_np, y_pred)
print(f"R² Score: {r2:.6f}")


Starting Symbolic Regression...
Start brute force search

Equation Found with mse: 0.049924410155743344
0.05*x_0/(0.05*x_1 + 0.02)

Fitting multiplication KAN to input data

Equation Found with mse: 2.4656472882213757e-15
0.54*x_0/(0.25*x_1**2 + 0.36)

--- Applying Substitutions ---
Replacing 'x_1' with '1.21338412886436*x_1'
Replacing 'x_0' with '1.18726578950488*x_0'
--------------------------------

New Expression: 0.359591551110038*x_0/(0.368075261044981*x_1**2 + 0.36)
Starting Backward Elimination.
Baseline Loss: 3.703395e-33 (Threshold: 3.740429e-33)
Backward Elimination Complete.
Final mse:  3.7033947641865206e-33
Final equation:  1.0*x_0/(x_1**2 + 1)

 recovered Expression: 1.0*x_0/(x_1**2 + 1)
R² Score: 1.000000
